In [209]:
import pandas as pd
import numpy as np
import re

In [210]:
df = pd.read_csv("../../datasets/raw/steam_games_requirements.csv")
df2 = pd.read_csv("../../datasets/steam_requirements_scraped.csv")

In [211]:
df2

,steam_appid,name,pc_requirements_minimum,pc_requirements_recommended
0,597170,Clone Drone in the Danger Zone,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN
1,42700,Call of Duty®: Black Ops,"<ul class=""bb_ul""><li><strong>OS *:</strong> W...",NaN
2,12210,Grand Theft Auto IV: The Complete Edition,"<ul class=""bb_ul""><li><strong>OS:</strong> Win...",NaN
3,400,Portal,<p><strong>Minimum: </strong>1.7 GHz Processor...,NaN
4,704450,Neverwinter Nights: Enhanced Edition,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
...,...,...,...,...
16185,912210,Achievement Collector: Cat,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN
16186,912140,SpaceBall in Cube,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN
16187,906470,Gravia,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN
16188,906430,Alive,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN


In [212]:
def clean_tags(text):
    if pd.isna(text):
        return None
    if '<br>' in text:
        text = re.sub('<br>', ',', text)
    if '\n' in text:
        text = re.sub('\n', ',', text)
    return " ".join(re.sub(r'<.*?>', ' ', text).split())

In [213]:
df2.dtypes

steam_appid                    int64
name                             str
pc_requirements_minimum          str
pc_requirements_recommended      str
dtype: object

In [214]:
df2.isna().sum()

steam_appid                        0
name                               8
pc_requirements_minimum          593
pc_requirements_recommended    15372
dtype: int64

In [215]:
df2.dropna(subset=['name', 'pc_requirements_minimum'], inplace=True) # The games here without names are DLCs, games without minimum requirements are DLCs, OSTs or Irrelevant.

In [216]:
df2 = df2[~df2['name'].str.contains(r'DLC|OST|SoundTrack')] # Cleaning DLCs and OSTs by string name

In [217]:
for i in df2['name']:
    if 'OST' in i:
        print(i)

In [218]:
df2.isna().sum()

steam_appid                        0
name                               0
pc_requirements_minimum            0
pc_requirements_recommended    14318
dtype: int64

In [219]:
df2['min_req'] = df2['pc_requirements_minimum'].apply(lambda x: clean_tags(x))

In [220]:
df2['rec_req'] = df2['pc_requirements_recommended'].apply(lambda x: clean_tags(x))

In [221]:
df2['min_req'][0]

'Minimum: , OS *: Windows 7 or newer, Processor: Modern quad-core (AMD FX-Series or newer, Intel Core i5 or faster), Memory: 2 GB RAM, Graphics: AMD Radeon HD 5770 or faster, Nvidia GeForce GT 640 or faster, Storage: 1 GB available space'

In [222]:
df2.drop(columns=['pc_requirements_minimum', 'pc_requirements_recommended'], inplace=True)

In [223]:
df2

,steam_appid,name,min_req,rec_req
0,597170,Clone Drone in the Danger Zone,"Minimum: , OS *: Windows 7 or newer, Processor...",NaN
1,42700,Call of Duty®: Black Ops,"OS *: Windows® Vista / XP / 7, Processor: Inte...",NaN
2,12210,Grand Theft Auto IV: The Complete Edition,"OS: Windows 10 (64-bit) , Processor: Intel Cor...",NaN
3,400,Portal,"Minimum: 1.7 GHz Processor, 512MB RAM, DirectX...",NaN
4,704450,Neverwinter Nights: Enhanced Edition,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and..."
...,...,...,...,...
16185,912210,Achievement Collector: Cat,"Minimum: , OS *: Windows 7, 8, 10, Processor: ...",NaN
16186,912140,SpaceBall in Cube,"Minimum: , OS *: Windows 7, Processor: 1000 MH...",NaN
16187,906470,Gravia,"Minimum: , OS *: Windows 7, Processor: 3rd Gen...",NaN
16188,906430,Alive,"Minimum: , OS *: Windows 7 or newer, Processor...",NaN
